# Phase 4: Feature Engineering

Builds the single merged, business-level feature table that Phase 5's three
modeling tasks (segmentation, risk classification, growth) will all read
from.

The key structural step here: `reviews_with_sentiment.csv` is one row per
**review**, but segmentation and risk classification need one row per
**business**. So the first thing this notebook does is aggregate the
review-level sentiment/text features up to the business level (e.g. average
sentiment per business), before merging with the business-level structured
data.

This notebook also defines the **proxy risk label** used in Phase 5b. To say
this clearly up front: the Yelp dataset has no real loan default or credit
outcome data. We use `is_open` (whether the business is still open on Yelp)
as a stand-in for financial distress — a business that has closed is a much
rougher, weaker signal than an actual default, but it's the best proxy this
dataset supports. This must be named explicitly as a proxy everywhere it's
used, never implied to be a real default label — see CLAUDE.md.

## Load both cleaned tables

In [1]:
import pandas as pd

business = pd.read_csv('../data/business_clean.csv')
reviews = pd.read_csv('../data/reviews_with_sentiment.csv', parse_dates=['date'])

business.shape, reviews.shape

((14560, 12), (967489, 17))

## Aggregate reviews to business level

For each business, compute:
- `review_count_actual` — how many reviews we actually have for this
  business in our filtered file (as a cross-check against the `review_count`
  field already in `business_clean.csv`, which comes from Yelp's own
  running total and could in principle drift from what we collected)
- `sentiment_compound_mean` / `sentiment_compound_std` — average sentiment,
  and how much it varies. Two businesses with the same average could look
  very different: one with consistently mediocre reviews (low std) vs. one
  with a mix of raves and disasters (high std) — that spread is itself a
  useful segmentation/risk signal
- `pct_negative_reviews` / `pct_positive_reviews` — share of a business's
  reviews that read as clearly negative/positive (VADER's own convention:
  `compound <= -0.05` and `>= 0.05` respectively, with the band in between
  treated as neutral)
- mean of the structural text features (word count, exclamation/question
  counts, caps ratio)

In [2]:
reviews['is_negative'] = reviews['sentiment_compound'] <= -0.05
reviews['is_positive'] = reviews['sentiment_compound'] >= 0.05

business_agg = reviews.groupby('business_id').agg(
    review_count_actual=('review_id', 'count'),
    sentiment_compound_mean=('sentiment_compound', 'mean'),
    sentiment_compound_std=('sentiment_compound', 'std'),
    pct_negative_reviews=('is_negative', 'mean'),
    pct_positive_reviews=('is_positive', 'mean'),
    text_word_count_mean=('text_word_count', 'mean'),
    text_exclamation_count_mean=('text_exclamation_count', 'mean'),
    text_question_count_mean=('text_question_count', 'mean'),
    text_caps_word_ratio_mean=('text_caps_word_ratio', 'mean'),
).reset_index()

# a business with exactly one review has undefined (NaN) std — treat as zero variance
business_agg['sentiment_compound_std'] = business_agg['sentiment_compound_std'].fillna(0)

business_agg.shape

(14560, 10)

### Cross-check `review_count_actual` against `business_clean.review_count`

These come from different sources (ours: counted directly from the review
file; Yelp's: a running total baked into the business record), so a small
gap is expected, but they should track closely — a big divergence would
mean something went wrong in Phase 2's filtering.

In [3]:
check = business[['business_id', 'review_count']].merge(business_agg[['business_id', 'review_count_actual']], on='business_id')
check['diff'] = check['review_count_actual'] - check['review_count']

print(check['diff'].describe())
print()
print('correlation:', check['review_count'].corr(check['review_count_actual']))

count    14560.000000
mean         2.150549
std          5.311804
min          0.000000
25%          0.000000
50%          0.000000
75%          2.000000
max        123.000000
Name: diff, dtype: float64

correlation: 0.9998630149833203


## Merge with the structured business data

Every business has at least one matched review (confirmed in Phase 2), so a
plain inner merge is safe here — no missing-review businesses to handle with
a left join + fillna.

Also add `category_count`: how many categories Yelp tags a business with
(e.g. "Restaurants, Food, Bakeries" → 3). This is a cheap structural
feature — more heavily-tagged businesses may be larger/more established.
Deeper use of the `categories` text itself (e.g. one-hot encoding the most
common categories) is left to the Phase 5a segmentation notebook, since
which categories are worth encoding is really a modeling-time choice.

In [4]:
business_features = business.merge(business_agg, on='business_id', how='inner')
business_features['category_count'] = business_features['categories'].str.split(',').str.len()

business_features.shape

(14560, 22)

## Define the proxy risk label

**`risk_proxy_closed`** = 1 if the business is closed on Yelp (`is_open` ==
0), 0 if still open. This is the target Phase 5b's classifier will predict.

**This is a proxy, not a real default label — full stop.** A business
closing on Yelp could mean bankruptcy, but could just as easily mean a
relocation, a rebrand under a new listing, an owner retiring, or Yelp data
simply going stale. There's no way to tell these apart in this dataset. This
limitation must be stated again in the Phase 7 write-up, and any results
from the Phase 5b model must be described as "predicting proxy business
closure," never as "predicting credit default" or "predicting risk" without
that qualifier.

In [5]:
business_features['risk_proxy_closed'] = (business_features['is_open'] == 0).astype(int)

business_features['risk_proxy_closed'].value_counts(normalize=True)

risk_proxy_closed
0    0.723558
1    0.276442
Name: proportion, dtype: float64

### ⚠️ Leakage warning for Phase 5b

`is_open` is kept in this saved file (rather than dropped) purely for
traceability/debugging — but it is **perfectly complementary** to
`risk_proxy_closed` by construction (`is_open == 1 - risk_proxy_closed` for
every row, confirmed below). If Phase 5b's feature matrix `X` is built as
"all columns except the target," `is_open` will leak in and hand the
classifier a trivial 100%-accuracy shortcut that has nothing to do with
sentiment or text signal.

**Phase 5b must explicitly exclude `is_open` (along with `risk_proxy_closed`
itself and identifier columns like `business_id`, `name`, `address`) from
`X`.** Restating this here so it isn't discovered as a bug after training.

In [6]:
assert (business_features['is_open'] == 1 - business_features['risk_proxy_closed']).all(), \
    'is_open and risk_proxy_closed are not perfectly complementary — investigate before trusting either'
print('confirmed: is_open is a perfect predictor of risk_proxy_closed by construction — exclude it from Phase 5b features')

confirmed: is_open is a perfect predictor of risk_proxy_closed by construction — exclude it from Phase 5b features


## On normalization/standardization

The plan calls for normalizing numeric features "needed later for
clustering." We're **not** baking a fixed scaling into this saved file,
for two reasons:

1. **Different Phase 5 tasks need different treatment.** K-means
   (Phase 5a) needs standardized features (its distance calculation is
   scale-sensitive), but tree-based models like XGBoost (Phase 5b) don't
   need scaling at all — feeding it pre-scaled features wouldn't hurt, but
   there's no reason to force it.
2. **Leakage risk for the classification task.** Standardizing means
   computing a mean/std from the data and subtracting/dividing by it. For
   Phase 5b, that mean/std must come from the *training split only* — if we
   fit it here on the full dataset before the train/test split even exists,
   information from the test set leaks into every training row's scaled
   values. Segmentation (5a) doesn't have this problem since it has no
   train/test split, but it's simpler and safer to apply scaling
   consistently at modeling time in each Phase 5 notebook rather than have
   one scaling decision made here serve two different needs.

To confirm scaling actually matters here, a quick before/after demonstration
on two columns (not saved — illustration only):

In [7]:
from sklearn.preprocessing import StandardScaler

demo_cols = ['review_count', 'sentiment_compound_mean']
print('before scaling:')
print(business_features[demo_cols].describe().loc[['mean', 'std']])

demo_scaled = StandardScaler().fit_transform(business_features[demo_cols])
print()
print('after scaling (mean should be ~0, std ~1 for both columns):')
print(pd.DataFrame(demo_scaled, columns=demo_cols).describe().loc[['mean', 'std']])

before scaling:
      review_count  sentiment_compound_mean
mean     64.297871                 0.551562
std     165.346818                 0.323251

after scaling (mean should be ~0, std ~1 for both columns):
      review_count  sentiment_compound_mean
mean  1.952040e-18            -9.540598e-17
std   1.000034e+00             1.000034e+00


## Save the merged feature table

One row per business, combining structured firmographic data, aggregated
review sentiment/text signal, and the proxy risk label — ready for Phase 5.

In [8]:
business_features.to_csv('../data/business_features.csv', index=False)
business_features.shape

(14560, 23)

## Final check

In [9]:
print(f'{len(business_features):,} businesses, {business_features.shape[1]} columns')
print()
print(list(business_features.columns))
business_features.head()

14,560 businesses, 23 columns

['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'categories', 'review_count_actual', 'sentiment_compound_mean', 'sentiment_compound_std', 'pct_negative_reviews', 'pct_positive_reviews', 'text_word_count_mean', 'text_exclamation_count_mean', 'text_question_count_mean', 'text_caps_word_ratio_mean', 'category_count', 'risk_proxy_closed']


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,...,sentiment_compound_mean,sentiment_compound_std,pct_negative_reviews,pct_positive_reviews,text_word_count_mean,text_exclamation_count_mean,text_question_count_mean,text_caps_word_ratio_mean,category_count,risk_proxy_closed
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107.0,39.955505,-75.155564,4.0,80,...,0.695723,0.530528,0.114943,0.873563,104.620690,1.540230,0.160920,0.002964,5,0
1,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,205 Race St,Philadelphia,PA,19106.0,39.953949,-75.143226,4.0,245,...,0.803010,0.453996,0.076000,0.920000,118.448000,1.560000,0.104000,0.003943,3,0
2,ROeacJQwBeh05Rqg7F6TCg,BAP,1224 South St,Philadelphia,PA,19147.0,39.943223,-75.162568,4.5,205,...,0.855654,0.292446,0.028846,0.961538,79.451923,1.086538,0.139423,0.004231,2,0
3,QdN72BWoyFypdGJhhI5r7g,Bar One,767 S 9th St,Philadelphia,PA,19147.0,39.939825,-75.157447,4.0,65,...,0.846458,0.244444,0.014493,0.985507,93.681159,1.347826,0.188406,0.005806,5,1
4,Mjboz24M9NlBeiOJKLEd_Q,DeSandro on Main,4105 Main St,Philadelphia,PA,19127.0,40.022466,-75.218314,3.0,41,...,0.304424,0.712733,0.317073,0.682927,88.097561,0.829268,0.219512,0.003735,4,1
